# Aligning serial breast cancer Xenium sections (squidpy + moscot)

In this notebook, we align two single cell resolution spatial transcriptomics datasets of serial breast cancer sections profiled by the Xenium technology.

The original notebook used STalign LDDMM with landmark points. Here we use `squidpy` with `moscot` optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (10, 8)

## Load data

In [ ]:
# Source: Rep1
fname = '../xenium_data/Xenium_FFPE_Human_Breast_Cancer_Rep1_cells.csv.gz'
df1 = pd.read_csv(fname)

coords_source = np.column_stack([df1['x_centroid'].values, df1['y_centroid'].values])
adata_source = ad.AnnData(X=np.zeros((len(coords_source), 1)), obs=df1)
adata_source.obsm['spatial'] = coords_source

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2)
ax.set_title('Xenium Rep1 (source)')

In [ ]:
# Target: Rep2
fname = '../xenium_data/Xenium_FFPE_Human_Breast_Cancer_Rep2_cells.csv.gz'
df2 = pd.read_csv(fname)

coords_target = np.column_stack([df2['x_centroid'].values, df2['y_centroid'].values])
adata_target = ad.AnnData(X=np.zeros((len(coords_target), 1)), obs=df2)
adata_target.obsm['spatial'] = coords_target

fig, ax = plt.subplots()
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.2, c='#ff7f0e')
ax.set_title('Xenium Rep2 (target)')

In [ ]:
# Overlay
fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='Rep1')
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.2, label='Rep2')
ax.legend(markerscale=10)

## Align using moscot

In [ ]:
sq.experimental.tl.align(
    adata_source,
    adata_target,
    method='optimal_transport',
    verbose=True,
)

## Visualize

In [ ]:
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='Rep1')
ax.scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='Rep1 aligned')
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.2, label='Rep2')
ax.legend(markerscale=10)

In [ ]:
# Also align target to source (reverse direction)
# With moscot, you can also run align on the target to get it into source space
adata_target_copy = adata_target.copy()
sq.experimental.tl.align(
    adata_target_copy,
    adata_source,
    method='optimal_transport',
    verbose=True,
)

aligned_tgt = adata_target_copy.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='Rep1')
ax.scatter(aligned_tgt[:, 0], aligned_tgt[:, 1], s=1, alpha=0.1, label='Rep2 aligned to Rep1')
ax.legend(markerscale=10)